# 05 — Forecasting Layer 3: Self-Monitoring

The model looks backward over its own predictions vs actuals and surfaces systematic bias patterns.

**Example output:**
> "Heads up: for the last 6 weeks, forecasts have consistently underestimated Thursday sales
> by ~8%. Recommend an upward adjustment to future Thursday predictions."

This layer is the portfolio differentiator — demonstrating a model that knows its own blind spots.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pickle
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from src.utils.db_connect import get_engine
from src.features.feature_engineering import build_features

engine = get_engine()
MODEL_DIR = Path('../src/models')
RAW_DIR = Path('../data/raw')
print('Connected')

## 1. Rolling-Window Forecast to Populate forecast_log

In [ ]:
df = pd.read_sql("""
    SELECT dd.*, ds.day_of_week, ds.event_flag, ds.temperature_index,
           ds.is_public_holiday, ds.trading_hours, ds.promo_flag
    FROM division_daily dd
    JOIN daily_sales ds ON dd.date = ds.date
    ORDER BY dd.date, dd.division_code
""", engine, parse_dates=['date'])
df = build_features(df)

FEATURE_COLS = [
    'division_code', 'temperature_index', 'trading_hours',
    'is_public_holiday', 'promo_flag', 'dow_num', 'month', 'day_of_year',
    'event_Arigato', 'event_Christmas', 'event_EOFY', 'event_Vivid', 'event_LongWeekend',
    'lag_7', 'lag_14', 'lag_28', 'rolling_7_mean', 'rolling_14_mean',
]

import xgboost as xgb

# Rolling window: train on weeks 1–n, predict week n+1
# Use the full dataset; minimum 26 weeks of history before first prediction
all_dates = sorted(df['date'].unique())
min_train_weeks = 26
min_train_date = all_dates[0] + pd.Timedelta(weeks=min_train_weeks)
predict_dates = [d for d in all_dates if d >= min_train_date]

log_rows = []
print(f'Running rolling-window forecasts over {len(predict_dates)} prediction dates...')

for i, target_date in enumerate(predict_dates):
    train_mask = df['date'] < target_date
    pred_mask  = df['date'] == target_date

    X_tr = df[train_mask][FEATURE_COLS].fillna(0)
    y_tr = df[train_mask]['sales_amt']
    X_pr = df[pred_mask][FEATURE_COLS].fillna(0)
    y_pr = df[pred_mask]['sales_amt']

    if len(X_pr) == 0 or len(X_tr) < 100:
        continue

    model = xgb.XGBRegressor(
        n_estimators=200, learning_rate=0.08, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        objective='reg:squarederror', random_state=42, verbosity=0
    )
    model.fit(X_tr, y_tr)

    # Quantile lower/upper
    xgb_lo = xgb.XGBRegressor(n_estimators=150, learning_rate=0.08, max_depth=5,
                                objective='reg:quantileerror', quantile_alpha=0.15,
                                random_state=42, verbosity=0)
    xgb_hi = xgb.XGBRegressor(n_estimators=150, learning_rate=0.08, max_depth=5,
                                objective='reg:quantileerror', quantile_alpha=0.85,
                                random_state=42, verbosity=0)
    xgb_lo.fit(X_tr, y_tr)
    xgb_hi.fit(X_tr, y_tr)

    preds    = model.predict(X_pr)
    lowers   = xgb_lo.predict(X_pr)
    uppers   = xgb_hi.predict(X_pr)
    actuals  = y_pr.values
    div_rows = df[pred_mask][['division_code', 'day_of_week', 'event_flag']].values

    forecast_date = target_date - pd.Timedelta(days=7)  # forecast made 1 week ahead

    for j in range(len(preds)):
        actual = actuals[j] if j < len(actuals) else None
        err_pct = ((actual - preds[j]) / actual) if actual and actual != 0 else None
        log_rows.append({
            'forecast_date': str(forecast_date.date()),
            'target_date':   str(target_date.date()),
            'division_code': int(div_rows[j][0]),
            'predicted_sales': round(float(preds[j]), 2),
            'lower_bound':     round(float(lowers[j]), 2),
            'upper_bound':     round(float(uppers[j]), 2),
            'actual_sales':    round(float(actual), 2) if actual else None,
            'error_pct':       round(float(err_pct), 4) if err_pct else None,
            'context_flags':   f'{div_rows[j][1]}/{div_rows[j][2]}',
        })

    if i % 30 == 0:
        print(f'  {i}/{len(predict_dates)} dates processed')

forecast_log = pd.DataFrame(log_rows)
forecast_log.to_csv(RAW_DIR / 'forecast_log.csv', index=False)

# Also load into DB
forecast_log.to_sql('forecast_log', engine, if_exists='replace', index=False, chunksize=10_000)
print(f'\nforecast_log: {len(forecast_log):,} rows saved.')

## 2. Rolling Bias Tracker

In [ ]:
fl = pd.read_sql("""
    SELECT fl.*, dd.division_name, ds.day_of_week
    FROM forecast_log fl
    JOIN division_daily dd ON fl.target_date = dd.date AND fl.division_code = dd.division_code
    JOIN daily_sales ds ON fl.target_date = ds.date
    WHERE fl.actual_sales IS NOT NULL
    ORDER BY fl.target_date, fl.division_code
""", engine, parse_dates=['target_date'])

print(f'Forecast log: {len(fl):,} rows with actuals')

# Signed error: positive = under-forecast, negative = over-forecast
fl['signed_error_pct'] = (fl['actual_sales'] - fl['predicted_sales']) / fl['actual_sales'] * 100

# Rolling 6-week mean bias per division × day_of_week
fl = fl.sort_values(['division_code', 'day_of_week', 'target_date'])
fl['rolling_6w_bias'] = (
    fl.groupby(['division_code', 'day_of_week'])['signed_error_pct']
    .transform(lambda x: x.rolling(6, min_periods=4).mean())
)

## 3. Detect Systematic Bias Flags

In [ ]:
BIAS_THRESHOLD = 5.0  # % — flag if |rolling bias| > 5%
CONSECUTIVE_WEEKS = 4

def count_consecutive_flagged(series, threshold):
    """Return count of most recent consecutive periods where |bias| > threshold."""
    flagged = (series.abs() > threshold).values
    count = 0
    for val in reversed(flagged):
        if val:
            count += 1
        else:
            break
    return count

bias_flags = []
for (div_code, dow), group in fl.groupby(['division_code', 'day_of_week']):
    g = group.sort_values('target_date').dropna(subset=['rolling_6w_bias'])
    if len(g) < CONSECUTIVE_WEEKS:
        continue

    consec = count_consecutive_flagged(g['rolling_6w_bias'], BIAS_THRESHOLD)
    if consec >= CONSECUTIVE_WEEKS:
        latest_bias = g['rolling_6w_bias'].iloc[-1]
        direction = 'underestimating' if latest_bias > 0 else 'overestimating'
        div_name = g['division_name'].iloc[-1]
        bias_flags.append({
            'division_code': div_code,
            'division_name': div_name,
            'day_of_week': dow,
            'consecutive_flagged_periods': consec,
            'latest_rolling_bias_pct': round(latest_bias, 2),
            'direction': direction,
            'alert': f"Heads up: {consec} consecutive weeks {direction} {dow} sales"
                     f" in {div_name} by ~{abs(latest_bias):.1f}%."
        })

bias_df = pd.DataFrame(bias_flags)
if len(bias_df) > 0:
    print(f'\nSystematic bias flags detected ({len(bias_df)}):')
    for _, row in bias_df.iterrows():
        print(f'  ⚠  {row["alert"]}')
else:
    print('No systematic biases detected above threshold.')

bias_df.to_csv(MODEL_DIR / 'bias_flags.csv', index=False)

## 4. Rolling Bias Chart by Division

In [ ]:
# Focus on top 4 revenue divisions
TOP_DIVS = [24, 34, 22, 32]
div_names = {24: "Women's Cut & Sewn", 34: "Men's Cut & Sewn",
              22: "Women's Bottoms", 32: "Men's Bottoms"}

fig = make_subplots(rows=2, cols=2, shared_xaxes=False,
                    subplot_titles=[div_names[d] for d in TOP_DIVS])
positions = [(1,1),(1,2),(2,1),(2,2)]

for i, div_code in enumerate(TOP_DIVS):
    row, col = positions[i]
    d = fl[(fl['division_code'] == div_code) & (fl['day_of_week'] == 'Thursday')].copy()
    d = d.sort_values('target_date').dropna(subset=['rolling_6w_bias'])

    # Flag regions
    flagged = d[d['rolling_6w_bias'].abs() > BIAS_THRESHOLD]

    fig.add_trace(go.Scatter(
        x=d['target_date'], y=d['rolling_6w_bias'],
        mode='lines', name=f'Div {div_code}', showlegend=(i==0),
        line=dict(color='#2196F3', width=2)
    ), row=row, col=col)

    if len(flagged) > 0:
        fig.add_trace(go.Scatter(
            x=flagged['target_date'], y=flagged['rolling_6w_bias'],
            mode='markers', name='Flagged', showlegend=(i==0),
            marker=dict(color='#F44336', size=6, symbol='circle')
        ), row=row, col=col)

    # Zero line
    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=row, col=col)
    fig.add_hline(y=BIAS_THRESHOLD, line_dash='dot', line_color='orange', row=row, col=col)
    fig.add_hline(y=-BIAS_THRESHOLD, line_dash='dot', line_color='orange', row=row, col=col)

fig.update_yaxes(ticksuffix='%')
fig.update_layout(
    title='Rolling 6-Week Forecast Bias by Division — Thursdays<br><sup>Orange dotted = ±5% threshold</sup>',
    height=600
)
fig.show()

## 5. Error Heatmap: Division × Day-of-Week

In [ ]:
error_heatmap = (
    fl.groupby(['division_name', 'day_of_week'])['signed_error_pct']
    .mean()
    .reset_index()
)

dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot = error_heatmap.pivot(index='division_name', columns='day_of_week', values='signed_error_pct')
pivot = pivot.reindex(columns=dow_order)

fig = px.imshow(
    pivot,
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    title='Mean Signed Forecast Error by Division × Day (Blue=Under-forecast, Red=Over-forecast)',
    labels=dict(color='Mean Error %')
)
fig.update_coloraxes(colorbar_ticksuffix='%')
fig.update_layout(height=650)
fig.show()